<a href="https://colab.research.google.com/github/abhuvan345/6thSem-ML-Lab/blob/main/1BM24CS403_LAB_3_LOGISTIC_REGRESSION_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import csv
import math
import random
import os
from google.colab import files

# -----------------------------
# Upload Files
# -----------------------------
uploaded = files.upload()

print("Files in directory:", os.listdir())

# -----------------------------
# Detect zoo-data file
# -----------------------------
zoo_file = None
for file in os.listdir():
    if "zoo" in file.lower() and "data" in file.lower():
        zoo_file = file

if zoo_file is None:
    raise FileNotFoundError("Zoo data file not found!")

print("Using file:", zoo_file)

# -----------------------------
# Load Dataset (Skip Header)
# -----------------------------
X = []
y = []

with open(zoo_file, "r") as file:
    reader = csv.reader(file)

    header = next(reader)   # 🔥 Skip header row

    for row in reader:
        features = list(map(float, row[1:-1]))  # Skip animal name
        label = int(row[-1]) - 1  # Convert 1–7 → 0–6

        X.append(features)
        y.append(label)

print("Samples Loaded:", len(X))

num_classes = len(set(y))
num_features = len(X[0])

# -----------------------------
# Normalize Features
# -----------------------------
def normalize(X):
    n = len(X[0])
    mins = [min(row[i] for row in X) for i in range(n)]
    maxs = [max(row[i] for row in X) for i in range(n)]

    for row in X:
        for i in range(n):
            if maxs[i] - mins[i] != 0:
                row[i] = (row[i] - mins[i]) / (maxs[i] - mins[i])
    return X

X = normalize(X)

# Add bias term
for i in range(len(X)):
    X[i] = [1] + X[i]

num_features += 1

# -----------------------------
# Train/Test Split
# -----------------------------
data = list(zip(X, y))
random.shuffle(data)

split = int(0.8 * len(data))
train_data = data[:split]
test_data = data[split:]

X_train = [d[0] for d in train_data]
y_train = [d[1] for d in train_data]
X_test = [d[0] for d in test_data]
y_test = [d[1] for d in test_data]

# -----------------------------
# Initialize Weights
# -----------------------------
weights = [[0.0]*num_features for _ in range(num_classes)]

# -----------------------------
# Softmax (Stable)
# -----------------------------
def softmax(z):
    max_z = max(z)
    exp_vals = [math.exp(v - max_z) for v in z]
    total = sum(exp_vals)
    return [v/total for v in exp_vals]

# -----------------------------
# Prediction
# -----------------------------
def predict_prob(x):
    logits = []
    for c in range(num_classes):
        z = sum(x[i] * weights[c][i] for i in range(num_features))
        logits.append(z)
    return softmax(logits)

def predict(x):
    probs = predict_prob(x)
    return probs.index(max(probs))

# -----------------------------
# Training
# -----------------------------
def train(X, y, lr=0.5, epochs=1000):
    m = len(X)

    for epoch in range(epochs):
        gradients = [[0.0]*num_features for _ in range(num_classes)]

        for i in range(m):
            probs = predict_prob(X[i])

            for c in range(num_classes):
                error = probs[c] - (1 if y[i] == c else 0)
                for j in range(num_features):
                    gradients[c][j] += error * X[i][j]

        for c in range(num_classes):
            for j in range(num_features):
                weights[c][j] -= (lr/m) * gradients[c][j]

        if epoch % 200 == 0:
            print("Epoch:", epoch)

# -----------------------------
# Accuracy
# -----------------------------
def accuracy(X, y):
    correct = 0
    for i in range(len(X)):
        if predict(X[i]) == y[i]:
            correct += 1
    return correct / len(X)

# -----------------------------
# Confusion Matrix
# -----------------------------
def confusion_matrix(X, y):
    matrix = [[0]*num_classes for _ in range(num_classes)]

    for i in range(len(X)):
        pred = predict(X[i])
        actual = y[i]
        matrix[actual][pred] += 1

    return matrix

# -----------------------------
# Train Model
# -----------------------------
train(X_train, y_train, lr=0.5, epochs=1000)

# -----------------------------
# Evaluate
# -----------------------------
print("\nTraining Accuracy:", accuracy(X_train, y_train))
print("Testing Accuracy:", accuracy(X_test, y_test))

print("\nConfusion Matrix:")
cm = confusion_matrix(X_test, y_test)
for row in cm:
    print(row)

Saving zoo-class-type.csv to zoo-class-type (5).csv
Files in directory: ['.config', 'zoo-class-type (3).csv', 'zoo-class-type.csv', 'zoo-class-type (2).csv', 'zoo-class-type (5).csv', 'zoo-class-type (4).csv', 'zoo-class-type (1).csv', 'zoo-data.csv', 'sample_data']
Using file: zoo-data.csv
Samples Loaded: 101
Epoch: 0
Epoch: 200
Epoch: 400
Epoch: 600
Epoch: 800

Training Accuracy: 1.0
Testing Accuracy: 0.8095238095238095

Confusion Matrix:
[6, 0, 0, 0, 0, 0, 0]
[0, 5, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 3, 0, 0, 0]
[0, 0, 1, 0, 1, 0, 0]
[0, 0, 0, 0, 0, 0, 2]
[0, 0, 1, 0, 0, 0, 2]
